In [6]:
## 1. Ingest: raw CSV -> typed Parquet (bronze)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType, BooleanType

spark = SparkSession.builder.appName("NYC Subway - Part 1 - ingest").getOrCreate()

RAW = "/home/jovyan/project/data/raw"
BRONZE = "/home/jovyan/project/data/bronze"

In [7]:
## 2. Ridership: 12 monthly files with the same 12 columns
ridership_schema = StructType([
    StructField("transit_timestamp", TimestampType()),
    StructField("transit_mode", StringType()),
    StructField("station_complex_id", StringType()),
    StructField("station_complex", StringType()),
    StructField("borough", StringType()),
    StructField("payment_method", StringType()),
    StructField("fare_class_category", StringType()),
    StructField("ridership", DoubleType()),
    StructField("transfers", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("georeference", StringType()),
])
rides = spark.read.option("header", True).schema(ridership_schema).csv(RAW + "/ridership")
rides.write.mode("overwrite").parquet(BRONZE + "/ridership")

In [8]:
## 3. Stations: we give the columns simple names
stations_schema = StructType([
    StructField("gtfs_stop_id", StringType()),
    StructField("station_id", IntegerType()),
    StructField("complex_id", IntegerType()),
    StructField("division", StringType()),
    StructField("line", StringType()),
    StructField("stop_name", StringType()),
    StructField("borough", StringType()),
    StructField("cbd", BooleanType()),
    StructField("daytime_routes", StringType()),
    StructField("structure", StringType()),
    StructField("gtfs_latitude", DoubleType()),
    StructField("gtfs_longitude", DoubleType()),
    StructField("north_direction_label", StringType()),
    StructField("south_direction_label", StringType()),
    StructField("ada", IntegerType()),
    StructField("ada_northbound", IntegerType()),
    StructField("ada_southbound", IntegerType()),
    StructField("ada_notes", StringType()),
    StructField("georeference", StringType()),
])
stations = spark.read.option("header", True).schema(stations_schema).csv(RAW + "/stations")
stations.write.mode("overwrite").parquet(BRONZE + "/stations")

In [9]:
## 4. Weather: 124 columns, we keep the 5 we need and give each one its type
# REPORT_TYPE appears twice in the file: Spark renames the copies REPORT_TYPE2 and REPORT_TYPE95 (identical, checked)
weather_raw = spark.read.option("header", True).csv(RAW + "/weather")
weather = weather_raw.select(
    col("DATE").cast("timestamp").alias("DATE"),
    col("REPORT_TYPE2").alias("REPORT_TYPE"),
    col("HourlyDryBulbTemperature").cast("double").alias("HourlyDryBulbTemperature"),
    col("HourlyPrecipitation").alias("HourlyPrecipitation"),
    col("HourlyPresentWeatherType").alias("HourlyPresentWeatherType"),
)
weather.write.mode("overwrite").parquet(BRONZE + "/weather")

In [11]:
## 5. Check: same number of rows as the raw files
print("ridership:", spark.read.parquet(BRONZE + "/ridership").count())
print("stations:", spark.read.parquet(BRONZE + "/stations").count())
print("weather:", spark.read.parquet(BRONZE + "/weather").count())

ridership: 29375687
stations: 496
weather: 11765
